___


___

# Análisis de sentimientos
Ahora que hemos visto los vectores de palabras, podemos comenzar a investigar el análisis de sentimientos. El objetivo es encontrar puntos en común entre los documentos, con el entendimiento de que los vectores *combinados* de manera similar deben corresponder a sentimientos similares.

Si bien el alcance del análisis de sentimientos es muy amplio, enfocaremos nuestro trabajo de la siguiente manera.

### 1. Clasificación de polaridad
No intentaremos determinar si una oración es objetiva o subjetiva, hecho u opinión. Más bien, solo nos importa si el texto expresa una opinión *positiva*, *negativa* o *neutral*.
### 2. Ámbito de nivel de documento
También intentaremos agregar todas las oraciones en un documento o párrafo, para llegar a una opinión general.
### 3. Análisis grueso
No intentaremos realizar un análisis detallado que determine el grado de positividad/negatividad. Es decir, no estamos tratando de adivinar cuántas estrellas otorgó un revisor, solo si la revisión fue positiva o negativa.

## Broad Steps:
* First, consider the text being analyzed. A model trained on paragraph-long movie reviews might not be effective on tweets. Make sure to use an appropriate model for the task at hand.
* Next, decide the type of analysis to perform. In the previous section on text classification we used a bag-of-words technique that considered only single tokens, or *unigrams*. Some rudimentary sentiment analysis models go one step further, and consider two-word combinations, or *bigrams*. In this section, we'd like to work with complete sentences, and for this we're going to import a trained NLTK lexicon called *VADER*.

## Módulo VADER de NLTK
VADER es un módulo NLTK que proporciona puntajes de sentimiento basados:
-  palabras utilizadas ("completamente" aumenta un puntaje, mientras que "ligeramente" lo reduce), 
-  mayúsculas y puntuación ("¡¡GRANDE!!!" es más fuerte que "genial"), 
-  y negaciones (palabras como "no es" y "no" afectan el resultado).
  
<br>Para ver el código fuente, visite https://www.nltk.org/_modules/nltk/sentiment/vader.html

**Descargue el léxico VADER.** Solo necesita hacer esto una vez.

In [1]:
import nltk
nltk.download('vader_lexicon')

/opt/anaconda3/envs/curso_nlp/lib/python3.7/site-packages/sklearn/utils/validation.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LARGE_SPARSE_SUPPORTED = LooseVersion(scipy_version) >= '0.14.0'


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/auhurtado/nltk_data...


True

In [3]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

/opt/anaconda3/envs/curso_nlp/lib/python3.7/site-packages/nltk/twitter/__init__.py:20: UserWarning: The twython library has not been installed. Some functionality from the twitter package will not be available.
  warnings.warn("The twython library has not been installed. "


`SentimentIntensityAnalyzer()` de NLTK toma una cadena y devuelve un diccionario de puntajes en cada una de las cuatro categorías:
* negativo
* neutral
* positivo
* compuesto *(calculado al normalizar las puntuaciones anteriores)*

In [6]:
a = 'This was a good movie.'
sid.polarity_scores(a)

{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}

In [7]:
a = 'This was the best, most awesome movie EVER MADE!!!'
sid.polarity_scores(a)

{'neg': 0.0, 'neu': 0.425, 'pos': 0.575, 'compound': 0.8877}

In [8]:
a = 'This was the worst film to ever disgrace the screen.'
sid.polarity_scores(a)

{'neg': 0.477, 'neu': 0.523, 'pos': 0.0, 'compound': -0.8074}

## Usa VADER para analizar las reseñas de Amazon
Para este ejercicio, aplicaremos `SentimentIntensityAnalyzer` a un conjunto de datos de 10,000 reseñas de Amazon. Al igual que nuestros conjuntos de datos de reseñas de películas, estos están etiquetados como "pos" o "neg". Al final, determinaremos la precisión de nuestro análisis de sentimiento con VADER.

In [9]:
import numpy as np
import pandas as pd

df = pd.read_csv('../datos/amazonreviews.tsv', sep='\t')
df.head()

,label,review
0,pos,Stuning even for the non-gamer: This sound tra...
1,pos,The best soundtrack ever to anything.: I'm rea...
2,pos,Amazing!: This soundtrack is my favorite music...
3,pos,Excellent Soundtrack: I truly like this soundt...
4,pos,"Remember, Pull Your Jaw Off The Floor After He..."


In [10]:
df['label'].value_counts()

neg    5097
pos    4903
Name: label, dtype: int64

### Limpiar los datos (opcional):
Recuerde que nuestro archivo moviereviews.tsv contenía registros vacíos. Verifiquemos si existe alguno en amazonreviews.tsv.

In [11]:
# ELIMINAR VALORES NaN Y CADENAS VACÍAS:
df.dropna(inplace=True)

blanks = [] # comenzar con una lista vacía

for i,lb,rv in df.itertuples():  # itera sobre el DataFrame
    if type(rv)==str:            # evita valores de NaN
        if rv.isspace():         # prueba 'review' por espacios en blanco
            blanks.append(i)     # agrega números de índice coincidentes a la lista

df.drop(blanks, inplace=True)

In [12]:
df['label'].value_counts()

neg    5097
pos    4903
Name: label, dtype: int64

En este caso no hubo registros vacíos. ¡Bien!

## Hagamos la primera revisión a través de VADER

In [13]:
sid.polarity_scores(df.loc[0]['review'])

{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'compound': 0.9454}

In [14]:
df.loc[0]['label']

'pos'

¡Excelente! Nuestra primera revisión fue etiquetada como "positiva" y obtuvo una puntuación compuesta positiva.

## Adición de puntuaciones y etiquetas al dataframe
Agregaremos columnas al DataFrame original para almacenar diccionarios de polarity_score, puntajes compuestos extraídos y nuevas etiquetas "pos/neg" derivadas del puntaje compuesto. Usaremos esta última columna para realizar una prueba de precisión.

In [15]:
df['scores'] = df['review'].apply(lambda review: sid.polarity_scores(review))

df.head()

,label,review,scores
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co..."
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co..."
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com..."
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com..."
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp..."


In [16]:
df['compound'] = df['scores'].apply(lambda score_dict: score_dict['compound'])

df.head()

,label,review,scores,compound
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co...",0.9454
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co...",0.8957
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com...",0.9858
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com...",0.9814
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp...",0.9781


In [17]:
df['comp_score'] = df['compound'].apply(lambda c: 'pos' if c >=0 else 'neg')

df.head()

,label,review,scores,compound,comp_score
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co...",0.9454,pos
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co...",0.8957,pos
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com...",0.9858,pos
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com...",0.9814,pos
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp...",0.9781,pos


## Informe sobre la precisión
Finalmente, usaremos scikit-learn para determinar qué tan cerca estuvo VADER de nuestras 10,000 etiquetas originales.

In [18]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [19]:
accuracy_score(df['label'],df['comp_score'])

0.7091

In [20]:
print(classification_report(df['label'],df['comp_score']))

              precision    recall  f1-score   support

         neg       0.86      0.51      0.64      5097
         pos       0.64      0.91      0.75      4903

   micro avg       0.71      0.71      0.71     10000
   macro avg       0.75      0.71      0.70     10000
weighted avg       0.75      0.71      0.70     10000



In [21]:
print(confusion_matrix(df['label'],df['comp_score']))

[[2623 2474]
 [ 435 4468]]


Esto nos dice que NLTK VADER identificó correctamente una reseña de Amazon como "positiva" o "negativa" aproximadamente el 71 % de las veces.
## A continuación: Proyecto de análisis de sentimiento